# 1. Getting started: symbols instead of numbers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/sympy-for-engineers/blob/main/notebooks/01_getting_started.ipynb)

NumPy answers "what is the number?". SymPy answers "what is the formula?".
This notebook covers the handful of ideas everything else builds on:

- exact numbers and symbols
- building, rearranging and simplifying expressions
- substituting values and getting a number out
- assumptions such as "this quantity is positive"
- the four mistakes almost every beginner makes

**Worked example:** the settling velocity of a small sphere (Stokes' law).

In [1]:
import sympy as sp

sp.init_printing(use_latex="mathjax")   # render results as typeset mathematics
print("SymPy", sp.__version__)

SymPy 1.14.0


## 1.1 Exact numbers

Floating-point numbers are approximations. SymPy keeps numbers exact until you ask for decimals.

In [2]:
print(0.1 + 0.2)                                   # ordinary Python floats
sp.Rational(1, 10) + sp.Rational(2, 10)            # exact

0.30000000000000004


3/10

In [3]:
sp.sqrt(8), sp.pi, sp.exp(1), sp.sin(sp.pi / 6)

(2⋅√2, π, ℯ, 1/2)

`evalf` (or the shorthand `.n`) converts to a decimal number with as many digits as you want.

In [4]:
sp.pi.evalf(40)

3.141592653589793238462643383279502884197

## 1.2 Symbols and expressions

A symbol is a name with no value. Declare what you know about it: here every quantity is
`positive`, which lets SymPy simplify things it otherwise could not (see 1.5).

In [5]:
rho, v, D, mu = sp.symbols("rho v D mu", positive=True)

Re = rho * v * D / mu      # the Reynolds number as a formula
Re

D⋅ρ⋅v
─────
  μ  

## 1.3 From formula to number: `subs` and `evalf`

Water at 20 °C flowing at 1.2 m/s in a 5 cm pipe:

In [6]:
values = {rho: 998, v: 1.2, D: 0.05, mu: 1.0e-3}
Re.subs(values)

59880.0000000000

The flow is turbulent ($Re \gg 4000$). Keep the numbers in a dictionary like `values`:
you can reuse it for every formula in the problem.

## 1.4 Rearranging expressions

SymPy does not guess which form you want. You choose the operation.

In [7]:
x, y = sp.symbols("x y")
expr = (x + 1) ** 3
sp.expand(expr)

 3      2          
x  + 3⋅x  + 3⋅x + 1

In [8]:
sp.factor(x**3 - 1)

        ⎛ 2        ⎞
(x - 1)⋅⎝x  + x + 1⎠

In [9]:
# one fraction <-> partial fractions
frac = 1 / (x * (x + 1))
sp.apart(frac), sp.together(1 / x - 1 / (x + 1))

⎛    1     1      1    ⎞
⎜- ───── + ─, ─────────⎟
⎝  x + 1   x  x⋅(x + 1)⎠

In [10]:
# simplify tries several strategies and returns the shortest result it finds
sp.simplify(sp.sin(x) ** 2 + sp.cos(x) ** 2), sp.simplify((x**2 - 1) / (x - 1))

(1, x + 1)

In [11]:
# collect groups powers of one variable: useful for reading off coefficients
a, b, c = sp.symbols("a b c")
sp.collect(sp.expand((a * x + b) * (x + c)), x)

   2                    
a⋅x  + b⋅c + x⋅(a⋅c + b)

## 1.5 Assumptions matter

Is $\sqrt{x^2} = x$? Only if $x \ge 0$. SymPy will not simplify unless it knows.

In [12]:
x_any = sp.symbols("x")
x_pos = sp.symbols("x", positive=True)
sp.sqrt(x_any**2), sp.sqrt(x_pos**2)

⎛   ____   ⎞
⎜  ╱  2    ⎟
⎝╲╱  x  , x⎠

Physical quantities such as mass, length, absolute temperature and viscosity are positive.
Saying so with `positive=True` avoids many "why didn't it simplify?" moments.
Other useful assumptions are `real=True` and `integer=True`.

## 1.6 Equations and solving for a variable

In Python `=` assigns and `==` compares objects. An equation is written with `sp.Eq`.

In [13]:
P, V, n, R, T = sp.symbols("P V n R T", positive=True)
ideal_gas = sp.Eq(P * V, n * R * T)
ideal_gas

P⋅V = R⋅T⋅n

In [14]:
sp.solve(ideal_gas, T)      # a list, because equations can have several solutions

⎡P⋅V⎤
⎢───⎥
⎣R⋅n⎦

## 1.7 Worked example: settling velocity of a small sphere

**Problem.** A small sphere of radius $R_p$ and density $\rho_p$ settles in a fluid of density
$\rho_f$ and viscosity $\mu$. Find its terminal velocity.

**By hand.** At terminal velocity the net weight equals the Stokes drag:

$$\tfrac{4}{3}\pi R_p^3(\rho_p-\rho_f)\,g = 6\pi\mu R_p v_t$$

**With SymPy.** Type the balance exactly as you would write it, and solve.

In [15]:
Rp, rho_p, rho_f, g, vt = sp.symbols("R_p rho_p rho_f g v_t", positive=True)

weight = sp.Rational(4, 3) * sp.pi * Rp**3 * (rho_p - rho_f) * g
drag = 6 * sp.pi * mu * Rp * vt

vt_expr = sp.solve(sp.Eq(weight, drag), vt)[0]
vt_expr

    2              
2⋅Rₚ ⋅g⋅(-ρ_f + ρₚ)
───────────────────
        9⋅μ        

**Numbers.** A 100 µm sand grain ($\rho_p = 2650$ kg/m³) in water:

In [16]:
sand = {Rp: 50e-6, rho_p: 2650, rho_f: 998, mu: 1.0e-3, g: 9.81}
vt_num = vt_expr.subs(sand)
vt_num        # m/s

0.00900340000000000

**Verify.** Stokes' law holds only for creeping flow, so check the particle Reynolds number.
Reuse the formula from 1.2 with the particle diameter and settling velocity.

In [17]:
Re_p = Re.subs({rho: sand[rho_f], v: vt_num, D: 2 * sand[Rp], mu: sand[mu]})
print(f"Re_p = {float(Re_p):.2f}")
assert Re_p < 1, "Stokes' law is not valid here"

Re_p = 0.90


$Re_p \approx 0.9$: just inside the range where Stokes' law is a fair approximation. A grain
twice as large would settle four times faster ($v_t \propto R_p^2$) and the check would fail.

A second check is the limit of a neutrally buoyant particle, which should not settle at all:

In [18]:
assert vt_expr.subs(rho_p, rho_f) == 0
vt_expr.subs(rho_p, rho_f)

0

## 1.8 Getting the result out

The same expression can be exported as LaTeX for a report or as code for a script.

In [19]:
print(sp.latex(vt_expr))
print(sp.pycode(vt_expr))

\frac{2 R_{p}^{2} g \left(- \rho_{f} + \rho_{p}\right)}{9 \mu}
(2/9)*R_p**2*g*(-rho_f + rho_p)/mu


## 1.9 Four classic mistakes

| Mistake | What happens | Do this instead |
| --- | --- | --- |
| `x^2` | `^` is "exclusive or" in Python | `x**2` |
| `1/3` | Python divides first and gives `0.333…` | `sp.Rational(1, 3)` or `sp.S(1)/3` |
| `y = x + 1` to state an equation | assigns a Python variable | `sp.Eq(y, x + 1)` |
| using `x` before defining it | `NameError` | `x = sp.symbols("x")` first |

In [20]:
x = sp.symbols("x")
x**2 * (1 / 3), x**2 * sp.Rational(1, 3)

⎛                       2⎞
⎜                   2  x ⎟
⎜0.333333333333333⋅x , ──⎟
⎝                      3 ⎠

## Exercises

1. Expand $(v + \Delta v)^2 - v^2$ and factor the result.
2. Solve Bernoulli's equation $P_1 + \tfrac12\rho v_1^2 = P_2 + \tfrac12\rho v_2^2$ for $v_2$.
   Why does SymPy return one solution when all symbols are positive and two when they are not?
3. For the sand grain above, find the radius at which $Re_p = 0.1$.
   Hint: substitute `vt_expr` into `Re`, then use `sp.solve`.
4. Print the ideal gas law solved for $V$ as LaTeX.